In [1]:
import os
import glob
import pandas as pd
from sdv.metadata import Metadata
from sdv.sequential import PARSynthesizer

import torch


# 1. Load all 8 CSV files and assign a unique series_id to each
csv_files = sorted(glob.glob("./data_clean/*.csv"))

dfs = []
for idx, file_path in enumerate(csv_files, start=1):
    temp_df = pd.read_csv(file_path)
    temp_df["series_id"] = idx
    dfs.append(temp_df)

combined_df = pd.concat(dfs, ignore_index=True)
print(f"Loaded {len(csv_files)} files with {len(combined_df)} total rows.")

# 2. Auto-detect metadata (using a dictionary) and set sequence key & time index
metadata = Metadata.detect_from_dataframes(data={"time_series": combined_df})

metadata.update_column(
    table_name="time_series",
    column_name="series_id",
    sdtype="id"
)
metadata.set_sequence_key(
    table_name="time_series",
    column_name="series_id"
)
metadata.set_sequence_index(
    table_name="time_series",
    column_name="Timer(ms)"
)

metadata.validate()

# 3. Train and save the synthesizer (pass the dictionary here too)
torch.backends.cudnn.enabled = False

synthesizer = PARSynthesizer(metadata, epochs=100, verbose=True)
synthesizer.fit({"time_series": combined_df})
synthesizer.save("./snapshots/par_synthesizer_8_series_no_windows.pkl")

# 4. Generate and export synthetic time series
synthetic_data = synthesizer.sample_sequences(num_sequences=50, sequence_length=150)

# Unpack if SDV returns a dictionary
if isinstance(synthetic_data, dict):
    synthetic_data = synthetic_data["time_series"]

os.makedirs("./data_features/synth", exist_ok=True)
synthetic_data.to_csv("./data_features/synth/synthetic_time_series_no_windows.csv", index=False)

print("Done! Saved synthetic_time_series.csv")

Loaded 8 files with 396881 total rows.


/home/akka/Documents/vscode_projects/SemesterProjectFall2026/.venv/lib/python3.12/site-packages/sdv/single_table/base.py:136: UserWarning: We strongly recommend saving the metadata using 'save_to_json' for replicability in future SDV versions.
  warnings.warn(
/home/akka/Documents/vscode_projects/SemesterProjectFall2026/.venv/lib/python3.12/site-packages/sdv/metadata/metadata.py:1226: UserWarning: The metadata lists columns in a different order than the data. This may result in the synthetic data having a different order.
Affected tables: 'time_series'.
  warnings.warn(
100%|██████████| 50/50 [02:31<00:00,  3.03s/it]

Done! Saved synthetic_time_series.csv
